# Curriculum Experiments — v8

## What changed from v5

Same model and training as v5, with:

- the three curricula run across **5 seeds at HIDDEN_SIZE = 5**, rather than a single network
- every learning curve shown as **mean ± std shaded bands** across seeds
- conflict accuracy plotted **two ways** (agreement with the stronger-side label, and the same result referenced to the 0.5 chance line)
- an **intensity-gap** breakdown (accuracy against how much the stronger cue wins by), using the intensities saved by `01_generate_trials_v8`
- the auditory-dominance rate and right/left split shown **across seeds**, to check whether a side preference is real or just symmetry-breaking

Loads from `./generated_trials_v8`.

## 1. Setup (MPS-enabled)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import time

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda"); print("Using GPU (CUDA):", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps"); print("Using GPU (MPS - Apple Silicon)")
else:
    device = torch.device("cpu"); print("WARNING: no GPU acceleration, using CPU (slower).")

DATA_DIR = Path("./generated_trials_v8")
MODEL_DIR = Path("./trained_models_v8"); MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
# Force CPU (the GRU is tiny; on Apple Silicon MPS can be slower for this size).
# Comment this cell out to use the device detected above.
import torch
device = torch.device("cpu")
torch.set_num_threads(torch.get_num_threads())
print("Forcing device:", device)

## 2. Load data and define subtask groups

In [ ]:
DET_SUBTASKS = ["det_absent","det_auditory_only","det_visual_only","det_multisensory"]
LOC_SUBTASKS = ["loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
                "loc_multisensory_same_L","loc_multisensory_same_R",
                "loc_conflict_audL_visR","loc_conflict_audR_visL"]
SUBTASKS = DET_SUBTASKS + LOC_SUBTASKS
TEST_ONLY = {"det_multisensory"}            # not trained
CONFLICT  = {"det_multisensory","loc_conflict_audL_visR","loc_conflict_audR_visL"}
LOC_CONFLICTS = ["loc_conflict_audL_visR","loc_conflict_audR_visL"]

# Colour each subtask by group: detection = oranges, localisation = blues.
_det_cols = plt.cm.Oranges(np.linspace(0.45, 0.92, len(DET_SUBTASKS)))
_loc_cols = plt.cm.Blues(np.linspace(0.35, 0.95, len(LOC_SUBTASKS)))
SUBTASK_STYLE = {}
for i, s in enumerate(DET_SUBTASKS):
    SUBTASK_STYLE[s] = dict(color=_det_cols[i], ls=("--" if s in CONFLICT else "-"))
for i, s in enumerate(LOC_SUBTASKS):
    SUBTASK_STYLE[s] = dict(color=_loc_cols[i], ls=("--" if s in CONFLICT else "-"))

def load_dataset(filename):
    d = np.load(DATA_DIR / filename, allow_pickle=True)
    out = {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64),
           "types": d["types"], "n_classes": int(d["n_classes"])}
    if "aud_int" in d:   # test set carries conflict intensities in v8
        out["aud_int"] = d["aud_int"].astype(np.float32)
        out["vis_int"] = d["vis_int"].astype(np.float32)
    return out

train = load_dataset("train.npz")
test  = load_dataset("test.npz")
print("Train:", train["X"].shape, " Test:", test["X"].shape)
print("Train class balance:", np.bincount(train["y"], minlength=4))
print("Subtasks in train:", sorted(set(train["types"])))

## 3. Split into detection-only and localisation-only subsets

`det_multisensory` is not in the training set, so the detection subset is the three trained detection subtasks; the localisation subset includes the conflict trials.

In [ ]:
is_detection   = np.array([t.startswith("det_") for t in train["types"]])
is_localisation = np.array([t.startswith("loc_") for t in train["types"]])

X_train_det, y_train_det = train["X"][is_detection],   train["y"][is_detection]
X_train_loc, y_train_loc = train["X"][is_localisation], train["y"][is_localisation]
print("Detection subset:",   X_train_det.shape, " classes:", np.bincount(y_train_det, minlength=4))
print("Localisation subset:", X_train_loc.shape, " classes:", np.bincount(y_train_loc, minlength=4))

## 4. Architecture and chosen hidden size

In [ ]:
class UnifiedGRU(nn.Module):
    def __init__(self, n_channels=4, hidden_size=8, n_classes=4):
        super().__init__()
        self.gru = nn.GRU(input_size=n_channels, hidden_size=hidden_size, batch_first=True)
        self.readout = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        x = x.transpose(1, 2)          # (B, channels, time) -> (B, time, channels)
        h, _ = self.gru(x)
        return self.readout(h)         # (B, time, n_classes)

HIDDEN_SIZE = 5          # set from 03_hidden_unit_sweep_v8
N_CLASSES = 4
BATCH_SIZE = 64
LR = 1e-3
print("Using HIDDEN_SIZE =", HIDDEN_SIZE)

## 5. Training function with per-subtask logging

In [ ]:
# Precompute test masks once.
TEST_MASKS = {s: (test["types"] == s) for s in SUBTASKS}

def make_loader(X, y, batch_size, shuffle):
    return DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y)),
                      batch_size=batch_size, shuffle=shuffle)

def train_phase(model, X_tr, y_tr, n_epochs, lr, batch_size, device, phase_label):
    # Trains the model and, every epoch, records test accuracy overall and per subtask.
    loader = make_loader(X_tr, y_tr, batch_size, True)
    X_te = torch.from_numpy(test["X"]).to(device)
    y_te = test["y"]
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    log = {s: [] for s in SUBTASKS}; log["overall"] = []
    for epoch in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb); B, T, C = logits.shape
            loss = loss_fn(logits.reshape(B*T, C), yb.unsqueeze(1).expand(B, T).reshape(B*T))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pred = model(X_te)[:, -1, :].argmax(-1).cpu().numpy()
        log["overall"].append(float((pred == y_te).mean()))
        for s in SUBTASKS:
            m = TEST_MASKS[s]
            log[s].append(float((pred[m] == y_te[m]).mean()))
    print("  [%s] done. final overall test acc = %.3f" % (phase_label, log["overall"][-1]))
    return log

def merge_logs(a, b):
    return {k: a[k] + b[k] for k in a}

## 6. Run the three curricula across 5 seeds

Each seed re-initialises the network (same dataset, different random init and batch order).

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
EPOCHS_PER_PHASE = 25
EPOCHS_UNIFIED = 50

all_logs = {"loc_then_det": [], "det_then_loc": [], "unified": []}
models   = {"loc_then_det": [], "det_then_loc": [], "unified": []}
BOUNDARY = {"loc_then_det": EPOCHS_PER_PHASE, "det_then_loc": EPOCHS_PER_PHASE, "unified": None}
t0 = time.time()

for seed in SEEDS:
    print("SEED", seed)
    # Condition 1: Localisation -> Detection
    torch.manual_seed(seed); np.random.seed(seed)
    m = UnifiedGRU(4, HIDDEN_SIZE, N_CLASSES).to(device)
    l1 = train_phase(m, X_train_loc, y_train_loc, EPOCHS_PER_PHASE, LR, BATCH_SIZE, device, "loc->det / loc")
    l2 = train_phase(m, X_train_det, y_train_det, EPOCHS_PER_PHASE, LR, BATCH_SIZE, device, "loc->det / det")
    all_logs["loc_then_det"].append(merge_logs(l1, l2)); models["loc_then_det"].append(m)

    # Condition 2: Detection -> Localisation
    torch.manual_seed(seed); np.random.seed(seed)
    m = UnifiedGRU(4, HIDDEN_SIZE, N_CLASSES).to(device)
    l1 = train_phase(m, X_train_det, y_train_det, EPOCHS_PER_PHASE, LR, BATCH_SIZE, device, "det->loc / det")
    l2 = train_phase(m, X_train_loc, y_train_loc, EPOCHS_PER_PHASE, LR, BATCH_SIZE, device, "det->loc / loc")
    all_logs["det_then_loc"].append(merge_logs(l1, l2)); models["det_then_loc"].append(m)

    # Condition 3: Unified
    torch.manual_seed(seed); np.random.seed(seed)
    m = UnifiedGRU(4, HIDDEN_SIZE, N_CLASSES).to(device)
    lU = train_phase(m, train["X"], train["y"], EPOCHS_UNIFIED, LR, BATCH_SIZE, device, "unified")
    all_logs["unified"].append(lU); models["unified"].append(m)

print("Total training time: %.1f min" % ((time.time()-t0)/60))

def stack(cond, key):
    return np.array([log[key] for log in all_logs[cond]])   # (n_seeds, n_epochs)

## 7. Total test accuracy across training (mean ± std)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cond_styles = {"loc_then_det": ("tab:green",  "Localisation -> Detection"),
               "det_then_loc": ("tab:purple", "Detection -> Localisation"),
               "unified":      ("black",      "Unified")}
for name, (color, lab) in cond_styles.items():
    A = stack(name, "overall"); epochs = np.arange(1, A.shape[1] + 1)
    mean, sd = A.mean(0), A.std(0)
    ax.plot(epochs, mean, color=color, lw=2.2, label=lab)
    ax.fill_between(epochs, mean - sd, mean + sd, color=color, alpha=0.18)
    if BOUNDARY[name] is not None:
        ax.axvline(BOUNDARY[name] + 0.5, color=color, ls="--", alpha=0.35, lw=1)
ax.axhline(0.25, color="gray", ls=":", alpha=0.6, label="4-class chance")
ax.set_xlabel("Epoch"); ax.set_ylabel("Overall test accuracy"); ax.set_ylim(0, 1.05)
ax.set_title("Total test accuracy across training (mean +/- std, %d seeds)" % len(SEEDS))
ax.grid(alpha=0.3); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

print("final overall test accuracy (mean +/- std):")
for name, (_, lab) in cond_styles.items():
    A = stack(name, "overall")[:, -1]
    print("  %-28s %.3f +/- %.3f" % (lab, A.mean(), A.std()))

## 8. Per-subtask training curves (mean ± std)

In [ ]:
def plot_subtask_curves(name, title):
    boundary = BOUNDARY[name]
    A_overall = stack(name, "overall"); epochs = np.arange(1, A_overall.shape[1] + 1)
    fig, (ax_det, ax_loc) = plt.subplots(2, 1, figsize=(11, 8.5), sharex=True)

    def draw(ax, subtasks, group_title):
        for s in subtasks:
            A = stack(name, s); mean, sd = A.mean(0), A.std(0)
            st = SUBTASK_STYLE[s]; is_conf = s in CONFLICT
            tag = (" (conflict, test-only)" if s == "det_multisensory"
                   else " (conflict)" if is_conf else "")
            ax.plot(epochs, mean, color=st["color"], ls=st["ls"],
                    lw=2.6 if is_conf else 1.8, label=s + tag)
            ax.fill_between(epochs, mean - sd, mean + sd, color=st["color"], alpha=0.12)
        m_ov, sd_ov = A_overall.mean(0), A_overall.std(0)
        ax.plot(epochs, m_ov, color="black", ls=":", lw=2, label="overall")
        ax.fill_between(epochs, m_ov - sd_ov, m_ov + sd_ov, color="black", alpha=0.10)
        if boundary is not None:
            ax.axvline(boundary + 0.5, color="red", alpha=0.4, lw=1.2)
        ax.axhline(0.25, color="gray", ls=":", alpha=0.5)
        ax.set_ylim(0, 1.05); ax.set_ylabel("Test accuracy")
        ax.set_title(group_title); ax.grid(alpha=0.3)
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)

    draw(ax_det, DET_SUBTASKS, title + "  -  Detection subtasks (mean +/- std, %d seeds)" % len(SEEDS))
    draw(ax_loc, LOC_SUBTASKS, "Localisation subtasks")
    if boundary is not None:
        ax_loc.text(boundary + 0.7, 0.02, "phase boundary", color="red", fontsize=8)
    ax_loc.set_xlabel("Epoch")
    plt.tight_layout(); plt.show()

plot_subtask_curves("loc_then_det", "Localisation -> Detection")
plot_subtask_curves("det_then_loc", "Detection -> Localisation")
plot_subtask_curves("unified",      "Unified")

## 9. Conflict choice breakdown (mean across seeds)

Only the **unified** column is interpretable; the sequential conditions collapse, so their conflict bars are a forgetting artefact, not an integration strategy.

In [ ]:
def conflict_props_per_seed(cond, subtask):
    m = TEST_MASKS[subtask]
    X_te = torch.from_numpy(test["X"]).to(device)
    props = []
    for model in models[cond]:
        model.eval()
        with torch.no_grad():
            pred = model(X_te)[:, -1, :].argmax(-1).cpu().numpy()
        c = np.bincount(pred[m], minlength=4).astype(float)
        props.append(c / c.sum())
    return np.array(props)  # (n_seeds, 4)

class_labels = ["no det (0)", "det (1)", "right (2)", "left (3)"]
class_colors = ["lightgray", "tab:orange", "tab:red", "tab:blue"]
conf_types = ["det_multisensory", "loc_conflict_audL_visR", "loc_conflict_audR_visL"]
conds = [("loc_then_det","Loc->Det"), ("det_then_loc","Det->Loc"), ("unified","Unified")]

fig, axes = plt.subplots(len(conf_types), len(conds), figsize=(13, 9), squeeze=False)
for r, ct in enumerate(conf_types):
    for cc, (name, lab) in enumerate(conds):
        P = conflict_props_per_seed(name, ct)
        mean, sd = P.mean(0), P.std(0)
        ax = axes[r][cc]
        ax.bar(class_labels, mean, yerr=sd, capsize=3, color=class_colors, edgecolor="black")
        ax.set_ylim(0, 1.0); ax.set_title(lab + "\n" + ct, fontsize=9)
        if cc == 0: ax.set_ylabel("proportion")
        for i, p in enumerate(mean): ax.text(i, p + 0.03, "%.2f" % p, ha="center", fontsize=8)
        ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
print("det_multisensory: proportion of class-1 (det) predictions = auditory-dominance rate.")
print("loc_conflict_*  : split across right(2)/left(3) = integration/bias readout.")

## 10. Conflict accuracy, two readings (unified)

Is the conflict curve agreement with the stronger-side label, which can legitimately exceed 0.5, or
should it be read against chance? Both readings are plotted.

In [ ]:
def unified_preds():
    X_te = torch.from_numpy(test["X"]).to(device)
    out = []
    for model in models["unified"]:
        model.eval()
        with torch.no_grad():
            out.append(model(X_te)[:, -1, :].argmax(-1).cpu().numpy())
    return out

preds_list = unified_preds()
agreement = {t: [] for t in LOC_CONFLICTS + ["pooled"]}
for pred in preds_list:
    pooled = np.isin(test["types"], LOC_CONFLICTS)
    agreement["pooled"].append(float((pred[pooled] == test["y"][pooled]).mean()))
    for t in LOC_CONFLICTS:
        m = TEST_MASKS[t]
        agreement[t].append(float((pred[m] == test["y"][m]).mean()))

labels = ["audL_visR", "audR_visL", "pooled"]
keys = LOC_CONFLICTS + ["pooled"]
means = [np.mean(agreement[k]) for k in keys]
stds  = [np.std(agreement[k])  for k in keys]
x = np.arange(len(keys))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].bar(x, means, yerr=stds, capsize=4, color="tab:blue")
axes[0].set_ylim(0, 1.02); axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel("Accuracy"); axes[0].set_title("A. Agreement with stronger-side label")
for xi, mm in zip(x, means): axes[0].text(xi, mm + 0.03, "%.2f" % mm, ha="center", fontsize=9)

axes[1].bar(x, means, yerr=stds, capsize=4, color="tab:blue")
axes[1].axhline(0.5, ls="--", color="tab:red", lw=1.8, label="chance (0.5)")
axes[1].set_ylim(0, 1.02); axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel("Accuracy"); axes[1].set_title("B. Same result, referenced to chance")
for xi, mm in zip(x, means): axes[1].text(xi, mm + 0.03, "+%.2f" % (mm - 0.5), ha="center", fontsize=9, color="tab:red")
axes[1].legend(loc="upper right", fontsize=9)
fig.suptitle("Conflict accuracy, two readings (unified, %d seeds)" % len(SEEDS), fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()
print("pooled conflict agreement: %.3f +/- %.3f" % (np.mean(agreement["pooled"]), np.std(agreement["pooled"])))

## 11. Intensity-gap breakdown (unified)

Accuracy on the localisation conflicts as a function of how much the stronger cue wins by. A rising
curve is the signature of reliability-weighting: easy when one cue clearly dominates, near chance on the near-ties.

In [ ]:
grid = np.linspace(0.15, 1.85, 5)
rows = []
for pred in preds_list:
    m = np.isin(test["types"], LOC_CONFLICTS)
    gap = np.abs(test["aud_int"][m] - test["vis_int"][m])
    correct = (pred[m] == test["y"][m]).astype(float)
    edges = np.quantile(gap, np.linspace(0, 1, 6)); edges[-1] += 1e-6
    centers, accs = [], []
    for i in range(5):
        sel = (gap >= edges[i]) & (gap < edges[i + 1])
        if sel.sum() == 0: continue
        centers.append(gap[sel].mean()); accs.append(correct[sel].mean())
    if len(centers) >= 2:
        rows.append(np.interp(grid, centers, accs))
rows = np.array(rows); mean, sd = rows.mean(0), rows.std(0)

plt.figure(figsize=(7.5, 5))
plt.plot(grid, mean, "-o", color="tab:blue", lw=2)
plt.fill_between(grid, mean - sd, mean + sd, color="tab:blue", alpha=0.18)
plt.axhline(0.5, ls="--", color="tab:red", lw=1.5, label="chance (0.5)")
plt.ylim(0, 1.02); plt.xlabel("Intensity gap |stronger - weaker|")
plt.ylabel("Conflict accuracy (agreement)")
plt.title("Reliability check: accuracy rises with the stronger cue's margin (mean +/- std, %d seeds)" % len(SEEDS))
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 12. Conflict choices across seeds

Is a side preference a real bias or just symmetry-breaking? `det_multisensory` is perfectly
left/right symmetric, so any consistent direction there should wash out across seeds.

In [ ]:
dom_rates = [float((pred[TEST_MASKS["det_multisensory"]] == 1).mean()) for pred in preds_list]
splits = {t: {"right": [], "left": []} for t in LOC_CONFLICTS}
for pred in preds_list:
    for t in LOC_CONFLICTS:
        m = TEST_MASKS[t]
        splits[t]["right"].append(float((pred[m] == 2).mean()))
        splits[t]["left"].append(float((pred[m] == 3).mean()))

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
axes[0].bar(range(len(dom_rates)), dom_rates, color="tab:orange", edgecolor="black")
axes[0].axhline(np.mean(dom_rates), ls="--", color="black", label="mean %.2f" % np.mean(dom_rates))
axes[0].set_title("det_multisensory\nauditory-dominance rate"); axes[0].set_xlabel("seed")
axes[0].set_ylabel("P(class = det)"); axes[0].set_ylim(0, 1.02); axes[0].legend(fontsize=8)
for ax, t in zip(axes[1:], LOC_CONFLICTS):
    r = np.array(splits[t]["right"]); l = np.array(splits[t]["left"]); xs = np.arange(len(r))
    ax.bar(xs - 0.18, r, width=0.36, color="tab:red", label="right")
    ax.bar(xs + 0.18, l, width=0.36, color="tab:blue", label="left")
    ax.set_title("%s\nright %.2f / left %.2f" % (t, r.mean(), l.mean()))
    ax.set_xlabel("seed"); ax.set_ylim(0, 1.02); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 13. Summary and save

In [ ]:
print("="*64)
print("SUMMARY: curriculum results (v8, %d seeds)" % len(SEEDS))
print("="*64)
nonconf = [s for s in SUBTASKS if s not in CONFLICT]
for name, lab in conds:
    ov = stack(name, "overall")[:, -1]
    nc = np.array([np.mean([all_logs[name][i][s][-1] for s in nonconf]) for i in range(len(SEEDS))])
    dom = conflict_props_per_seed(name, "det_multisensory")[:, 1]
    print("\n%s:" % lab)
    print("  overall test acc:            %.3f +/- %.3f" % (ov.mean(), ov.std()))
    print("  mean non-conflict subtask:   %.3f +/- %.3f" % (nc.mean(), nc.std()))
    print("  auditory-dominance rate:     %.3f +/- %.3f  (det_multisensory -> class 1)" % (dom.mean(), dom.std()))

for i, model in enumerate(models["unified"]):
    torch.save(model.state_dict(), MODEL_DIR / ("unified_seed%d.pt" % i))
np.savez(MODEL_DIR / "curriculum_logs_v8.npz", all_logs=np.array(all_logs, dtype=object),
         seeds=np.array(SEEDS))
print("\nSaved unified models + logs to", MODEL_DIR)